# Development of the pretraining `LightningModule`.

## Setup

In [1]:
from omegaconf import OmegaConf
from pathlib import Path

import torch
import lightning as L

import wandb

from fm4tag.datamodules.datamodule import CatConDataModule
from fm4tag.utils import instantiate_callbacks, instantiate_loggers

#### Configuration

In [2]:
CONFIG_DIR = Path('/storage3/DSIP/rriva/research/fm4tag-review/src/fm4tag/configs')
CONFIG = 'pretraining_test_260630'

config_path = CONFIG_DIR / f"{CONFIG}.yaml"

cfg = OmegaConf.load(config_path)

#### Precision and seed

In [3]:
torch.set_float32_matmul_precision('medium') # options: 'medium', 'high', 'highest'

if cfg.get('seed'):
    L.seed_everything(cfg.get('seed'), workers=True)

Seed set to 42


In [4]:
help(L.seed_everything)

Help on function seed_everything in module lightning.fabric.utilities.seed:

seed_everything(
    seed: Optional[int] = None,
    workers: bool = False,
    verbose: bool = True
) -> int
    Function that sets the seed for pseudo-random number generators in: torch, numpy, and Python's random module.
    In addition, sets the following environment variables:

    - ``PL_GLOBAL_SEED``: will be passed to spawned subprocesses (e.g. ddp_spawn backend).
    - ``PL_SEED_WORKERS``: (optional) is set to 1 if ``workers=True``.

    Args:
        seed: the integer value seed for global random state in Lightning.
            If ``None``, it will read the seed from ``PL_GLOBAL_SEED`` env variable. If ``None`` and the
            ``PL_GLOBAL_SEED`` env variable is not set, then the seed defaults to 0. If seed is
            not in bounds or cannot be cast to int, a ValueError is raised.
        workers: if set to ``True``, will properly configure all dataloaders passed to the
            Trainer wit

#### Callbacks and loggers

In [5]:
callbacks = instantiate_callbacks(cfg.callbacks)
logger = instantiate_loggers(cfg.loggers)
print(f"Instantiated {len(callbacks)} callbacks: {[type(c).__name__ for c in callbacks]}")
print(f"Instantiated {len(loggers)} loggers: {[type(l).__name__ for l in loggers]}")

Instantiated 5 callbacks: ['ModelSummary', 'TQDMProgressBar', 'ModelCheckpoint', 'EarlyStopping', 'BackboneFinetuning']


NameError: name 'loggers' is not defined

#### Trainer

In [ ]:
trainer_kwargs = OmegaConf.to_container(cfg.trainer, resolve=True)
print(f"Trainer kwargs: {trainer_kwargs}")

Trainer kwargs: {'max_epochs': 100, 'accelerator': 'gpu', 'devices': 1, 'precision': 'bf16-mixed', 'log_every_n_steps': 100}


In [ ]:
n = 4

trainer = L.Trainer(
    accelerator='cpu',
    devices=n,
    precision=cfg.get('precision', 'bf16-mixed'),
    max_epochs=cfg.get('max_epochs', 1),
    log_every_n_steps=cfg.get('log_every_n_steps', 50),
    logger=logger,
    callbacks=callbacks,
    strategy='ddp_notebook' if n > 1 else 'auto',
    #limit_train_batches=1
)

/storage3/DSIP/rriva/research/fm4tag-review/.venv/lib/python3.13/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /storage3/DSIP/rriva/research/fm4tag-review/.venv/li ...
Using bfloat16 Automatic Mixed Precision (AMP)
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [6]:
dl_cfg = cfg.dataloader

In [8]:
datamodule = CatConDataModule(
    train_dataset_path=cfg.train_dataset_path if cfg.phase == 'finetune' else cfg.pretrain_dataset_path,
    val_dataset_path=cfg.val_dataset_path,
    test_dataset_path=cfg.test_dataset_path,
    variables=cfg.variables,
    global_object=cfg.global_object,
    constituent_objects=list(cfg.constituent_objects),
    norm_dict_path=cfg.get('norm_dict_path'),
    class_dict_path=cfg.get('class_dict_path'),
    batch_size=dl_cfg.batch_size,
    num_workers=dl_cfg.num_workers,
    prefetch_factor=dl_cfg.get('prefetch_factor', 2),
    pin_memory=dl_cfg.get('pin_memory', True),
    persistent_workers=dl_cfg.get('persistent_workers', False)
)